In [7]:
import os
import glob
import argparse
import json
import numpy as np
import cv2
import torch
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_lightning import seed_everything
from src.test.test_codec import process_images
from test_utils import calculate_metrics_batch
from models.util import create_model, load_state_dict
from models.ddim_hacked import DDIMSampler
import torch
import einops

def tensor_to_image(tensor):
    """ Convert a tensor to a numpy image for visualization. """
    tensor = tensor.detach().cpu().numpy()
    tensor = (tensor - tensor.min()) / (tensor.max() - tensor.min())  # Normalize
    tensor = np.transpose(tensor, (1, 2, 0))  # Convert from CHW to HWC
    return tensor

# model = create_model('experiment_fdn/local_v15.yaml').cpu()
# model.load_state_dict(load_state_dict('experiment_fdn/local_ckpt/local-best-checkpoint-v1.ckpt', location="cuda"))
# model = model.cuda()

local_adapter = model.local_adapter
feature_extractor = local_adapter.feature_extractor
pre_extractor = feature_extractor.pre_extractor

In [8]:

op_path = "data/UVG/Beauty/conditions/optical_flow/im00002.png"
frame_path = "data/UVG/Beauty/conditions/quality_8/im00002.png"
W, H = 512, 512
num_samples = 1   
op_image = cv2.cvtColor(cv2.imread(op_path), cv2.COLOR_BGR2RGB)
frame_image = cv2.cvtColor(cv2.imread(frame_path), cv2.COLOR_BGR2RGB)

# Resize images
op_image = cv2.resize(op_image, (W, H))
frame_image = cv2.resize(frame_image, (W, H))

# Ensure images are in HWC format
op_map = np.asarray(op_image, dtype=np.float32)
frame_map = np.asarray(frame_image, dtype=np.float32)

print(f"Optical Flow Shape: {op_map.shape}")   # Should be (H, W, 3)
print(f"Frame Image Shape: {frame_map.shape}") # Should be (H, W, 3)

# Concatenate along channel axis (H, W, 6)
detected_maps = np.concatenate([op_map, frame_map], axis=2)

# Normalize and convert to tensor
local_control = torch.from_numpy(detected_maps.copy()).float().cuda() / 255.0

# Expand to match batch size
local_control = torch.stack([local_control for _ in range(num_samples)], dim=0)

# Rearrange to (batch, channels, height, width)
local_control = einops.rearrange(local_control, "b h w c -> b c h w").clone()

Optical Flow Shape: (512, 512, 3)
Frame Image Shape: (512, 512, 3)


In [ ]:
with torch.no_grad():
    op = local_control[:, :3, :, :]  # First 3 channels (Optical Flow)
    frame = local_control[:, 3:, :, :]  # Last 3 channels (Frame)

    # Apply pre-extractor
    local_features_op = pre_extractor(op, None)
    local_features_frame = pre_extractor(frame, None)

    # Apply noise to the extracted optical flow features
    local_features_op_noisy = feature_extractor.add_uniform_noise(local_features_op, noise_level=0.1)

# Convert tensors to images
op_img = tensor_to_image(op[0])  # Take the first batch
frame_img = tensor_to_image(frame[0])
local_features_op_img = tensor_to_image(local_features_op[0])
local_features_frame_img = tensor_to_image(local_features_frame[0])
local_features_op_noisy_img = tensor_to_image(local_features_op_noisy[0])

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(op_img)
axes[0, 0].set_title("Original Optical Flow")
axes[0, 0].axis("off")

axes[0, 1].imshow(local_features_op_img)
axes[0, 1].set_title("Extracted Optical Flow Features")
axes[0, 1].axis("off")

axes[0, 2].imshow(local_features_op_noisy_img)
axes[0, 2].set_title("Noisy Optical Flow Features")
axes[0, 2].axis("off")

axes[1, 0].imshow(frame_img)
axes[1, 0].set_title("Original Frame")
axes[1, 0].axis("off")

axes[1, 1].imshow(local_features_frame_img)
axes[1, 1].set_title("Extracted Frame Features")
axes[1, 1].axis("off")

axes[1, 2].axis("off")  # Empty space

plt.tight_layout()
plt.show()

## Caption BPP Calculation

In [1]:
import zlib

def compress_text(input_text):
    """Compress the input text to bytes using zlib."""
    input_bytes = input_text.encode('utf-8')
    return zlib.compress(input_bytes, level=zlib.Z_BEST_COMPRESSION)

def calculate_bpp(compressed_data, num_pixels, bytes=True, num_bytes=None):
    """Calculate BPP given the compressed text and number of pixels."""
    scaling_factor = 8 if bytes else 1
    if num_bytes:
        return num_bytes * scaling_factor / num_pixels
    return len(compressed_data) * scaling_factor / num_pixels

# Video details with captions
video_details = {
    "Beauty": {
        "prompt": "A beautiful blonde girl with pink lipstick with black background",
        "path": "Beauty"
    },
    "Jockey": {
        "prompt": "The image features a man riding a brown horse, galloping through a grassy field. The man is wearing a yellow shirt and is skillfully guiding the horse.",
        "path": "Jockey"
    },
    "Bosphorus": {
        "prompt": "The image features a man and a woman sitting together on a boat in the water. They are both wearing ties, suggesting a formal or semi-formal occasion.",
        "path": "Bosphorus"
    }
}

# Image dimensions
H, W = 512, 512  # Assuming images are 512x512 pixels
num_pixels = H * W

# Compute BPP for each caption
bpp_results = {}
for video_name, details in video_details.items():
    prompt = details["prompt"]
    compressed_text = compress_text(prompt)
    bpp_text = calculate_bpp(compressed_text, num_pixels)
    bpp_results[video_name] = bpp_text
    print(f"{video_name} - BPP for caption: {bpp_text:.4f}")

# Display results
print("\nFinal BPP for each caption:")
for video, bpp in bpp_results.items():
    print(f"{video}: {bpp:.4f}")


Beauty - BPP for caption: 0.0020
Jockey - BPP for caption: 0.0035
Bosphorus - BPP for caption: 0.0034

Final BPP for each caption:
Beauty: 0.0020
Jockey: 0.0035
Bosphorus: 0.0034


## Intra Coded Bpp Reduction 

## Optical Flow Sparse Experiments